# CHELSA-TraCE21k → yearly-mean netCDF
Converts CHELSA-TraCE21k monthly temperature GeoTIFFs (`tasmin` / `tasmax`)
into one netCDF per variable, holding the **yearly mean** at each
**100-year (centennial) time step**, at full ~1 km resolution — then derives a
**`tasmean`** netCDF from those two outputs.

The notebook has two stages:
1. **Build** — GeoTIFFs → `tasmin.nc` and `tasmax.nc` (the main loop below).
2. **Derive** — `tasmin.nc` + `tasmax.nc` → `tasmean.nc` (the tasmean cell).

## Configuration
- `VARIABLES` — which variables to process (e.g. `["tasmin"]`).
- `BASE_DIR` / `INPUT_DIR` / `OUTPUT_DIR` — input and output locations.
- `SCALE_FACTOR = 0.1` — CHELSA stores temperature as Kelvin × 10; this rescales it.
- `COARSEN_FACTOR` — `None` for full resolution, or an int for N×N block-mean downsampling.
- `LONG_NAMES` — per-variable metadata description, looked up by variable name.
- `CHUNKS = {"x": 4500, "y": 2250}` — chunk size for the streaming write.
- `dask.config.set(num_workers=4)` — caps parallel tasks.

## Helpers
- **`index_to_year(time_index)`**:
    - Maps a CHELSA time index to a calendar year.
    - Anchor `t = 20 → 1990 CE`, each step = 100 years.
    - Each filename looks like `CHELSA_TraCE21k_tasmax_<MM>_<TTT>_V.1.0.tif`, where `MM` is the month (`01`–`12`) and `TTT` is a CHELSA time index. The script reads the time index from the filename and converts it to a calendar year
    - Mapping:
    | TTT     | Year (CE) | Comment             |
    |---------|-----------|---------------------|
    | `0020`  | 1990      | most recent step    |
    | `0000`  | -10       | ~ year 0            |
    | `-001`  | -110      | 100 years older     |
    | `-200`  | -20010    | ~ 22 ka BP, ≈ LGM   |
    - **DISCLAIMER**: 
    The mapping `year = 1990 − (20 − TTT) × 100` is a hypothesis. CHELSA's
    official documentation does not explicitly publish the index-to-year table.
    The hypothesis is consistent with:
        - the metadata field `frequency=centennial` in your `.tif` files,
        - the paper's claim of ~21,000 years of coverage,
        - the file count (221 timesteps × 100 years ≈ 22 ka).
- **`group_files_by_year(variable)`**:
    - globs the input folder, matches each filename against a regex to pull out the month and time index, converts the index to a year, and returns a dict mapping each year to its list of monthly file paths.
- **`lazy_yearly_mean(monthly_files)`**:
    - opens the 12 monthly rasters, optionally coarsens them, stacks along a `month` dimension and averages to one `(y, x)` field, applies the scale factor, and casts to `float32`.

## Main loop (build stage)
For each variable:
1. Group its files by year and stop early if none are found.
2. For each year (in chronological order), check it has 12 months, build the
   yearly-mean field, and stamp it with a mid-year timestamp on a new `time` axis.
3. Concatenate all years along `time`, rename the spatial axes to `lat`/`lon`,
   and attach CF-style metadata (`long_name`, `units`, `source`, `comment`).
4. Write to netCDF — gzip-compressed, `float32`, with `time` as an unlimited
   (appendable) dimension.

## tasmean (derive stage)
Computes `tasmean = (tasmin + tasmax) / 2` from the two build outputs. Both
inputs are already in Kelvin, so it is a plain average with no rescaling.

The inputs are ~125 GB each — too large to hold in RAM — so the cell streams
through them block by block, just like the build stage:
- **`chunks={}`** inherits each file's own on-disk chunking, so reads stay
  aligned to the stored chunk grid (no read amplification).
- **`MULT`** fuses whole on-disk chunks into bigger work units (an exact
  multiple of the native chunk never splits one). Bigger `MULT` = fewer, fatter
  blocks = more RAM in flight = faster.
- **`scheduler="threads", num_workers=8`** decompresses, averages, and
  recompresses many blocks in parallel while a single writer streams results to
  disk. Peak RAM ≈ `num_workers × block size + write backlog`; raise
  `num_workers` and/or `MULT` to push it toward the RAM budget (~16 GB target on
  28 GB hardware).
- The CRS (`spatial_ref` coordinate, written by rioxarray) is carried over to
  the output so it stays georeferenced.

## Laziness & memory
Every step before a write is **lazy**: `open_rasterio(chunks=...)` /
`open_dataset(chunks=...)`, `concat`, `mean`, `* SCALE_FACTOR`, `astype`, the
`(a + b) * 0.5` average, and `expand_dims` only build a dask task graph — no
data is read or computed. Data is materialized **chunk by chunk** only during
`to_netcdf`, so peak RAM is set by the chunk size (build stage ≈ 40 MB per
chunk: 4500 × 2250 × 4 bytes) times the number of workers, not by the full
array. The `.astype("float32")` after scaling/averaging matters because
multiplying by a Python float (`0.1`, `0.5`) upcasts to float64; casting back
keeps data at 4 bytes/pixel.

## Output
Three netCDF files in `OUTPUT_DIR`, each with dimensions `(time, lat, lon)` and
self-describing via metadata in the header:
- `tasmin.nc` — yearly mean of daily minimum temperature (build stage).
- `tasmax.nc` — yearly mean of daily maximum temperature (build stage).
- `tasmean.nc` — yearly mean of daily mean temperature (derive stage).

In [1]:
import re
from pathlib import Path
import cftime
import dask
import rioxarray
import xarray as xr
from dask.diagnostics import ProgressBar
from tqdm import tqdm
from tqdm.dask import TqdmCallback
import subprocess
import tarfile
from tempfile import TemporaryDirectory

In [2]:
def index_to_year(time_index):
    """Converts CHELSA centennial time index to calendar year. Anchor: t=20 -> 1990 CE."""
    return 1990 - (20 - time_index) * 100


def group_files_by_year(variable):
    """Maps each calendar year to its list of monthly GeoTIFF paths."""
    pattern = re.compile(rf"CHELSA_TraCE21k_{variable}_(\d{{2}})_(-?\d+)_V\.1\.0\.tif")
    files_by_year = {}
    for path in INPUT_DIR.glob(f"CHELSA_TraCE21k_{variable}_*.tif"):
        match = pattern.match(path.name)
        if match:
            year = index_to_year(int(match.group(2)))
            files_by_year.setdefault(year, []).append(path)
    return files_by_year


def lazy_yearly_mean(monthly_files):
    """Lazy 12-month mean for one year. Nothing is computed until the write."""
    months = []
    for path in monthly_files:
        da = rioxarray.open_rasterio(path, chunks=CHUNKS, masked=True).squeeze("band", drop=True)
        if COARSEN_FACTOR:
            da = da.coarsen(x=COARSEN_FACTOR, y=COARSEN_FACTOR, boundary="trim").mean()
        months.append(da)

    yearly_mean = xr.concat(months, dim="month").mean(dim="month")
    # *0.1 upcasts to float64; cast back so chunks and on-disk data stay 4 bytes/pixel.
    return (yearly_mean * SCALE_FACTOR).astype("float32")

In [3]:
VARIABLES = [
    "tasmin",
    "tasmax"
]

BASE_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial"
INPUT_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "data"
OUTPUT_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE_FACTOR = 0.1     # CHELSA stores temperature as Kelvin × 10
COARSEN_FACTOR = None  # set to an int for block-mean downsampling (lower resolution and smaller file size)

LONG_NAMES = {
    "tasmax": "Yearly mean of daily maximum near-surface air temperature",
    "tasmin": "Yearly mean of daily minimum near-surface air temperature",
}

# Chunk = unit of work for the streaming write. Each chunk is computed, written,
# then freed, so chunk size sets peak RAM (not the full array). 4500×2250 float32 ≈ 40 MB.
CHUNKS = {"x": 4500, "y": 2250}

# Cap parallel tasks so peak RAM stays bounded during the streaming write.
dask.config.set(num_workers=4)

In [4]:
for variable in VARIABLES:
    print(f"Processing variable: {variable}")

    files_by_year = group_files_by_year(variable)
    if not files_by_year:
        raise SystemExit(f"No matching .tif files found in {INPUT_DIR}")

    slices = []
    for year in tqdm(sorted(files_by_year), desc="Building yearly means"):
        monthly_files = files_by_year[year]
        if len(monthly_files) != 12:
            raise ValueError(f"Year {year} has {len(monthly_files)} months, expected 12")

        timestamp = cftime.DatetimeProlepticGregorian(year, 7, 1) # builds a timestamp at July 1st of that year.
        slices.append(lazy_yearly_mean(monthly_files).expand_dims(time=[timestamp]))

    combined = xr.concat(slices, dim="time").rename({"x": "lon", "y": "lat"})
    combined.name = variable
    combined.attrs = {
        "long_name": LONG_NAMES[variable],
        "units": "K",
        "source": "CHELSA-TraCE21k v1.0 (centennial)",
        "comment": "Each timestep is a centennial climatology averaged over 12 months.",
    }
    if COARSEN_FACTOR:
        combined.attrs["spatial_processing"] = (f"Spatially downsampled by factor {COARSEN_FACTOR} (block-mean)")

    output_path = OUTPUT_DIR / f"{variable}.nc"
    print(f"Writing {output_path}")
    with ProgressBar():
        combined.to_netcdf(
            output_path,
            encoding={variable: {"zlib": True, "complevel": 4, "dtype": "float32"}},
            unlimited_dims=["time"],
        )
    print("Done.")

Processing variable: tasmin


Building yearly means: 100%|███████████████████████████████████████████████████████████████| 221/221 [00:16<00:00, 13.39it/s]


Writing /media/eskilsg/INTENSO/paleoclimate_model_comparison/data/CHELSA-TraCE21k-centennial/output/tasmin.nc
[########################################] | 100% Completed | 4hr 58m
Done.
Processing variable: tasmax


Building yearly means: 100%|███████████████████████████████████████████████████████████████| 221/221 [02:02<00:00,  1.81it/s]


Writing /media/eskilsg/INTENSO/paleoclimate_model_comparison/data/CHELSA-TraCE21k-centennial/output/tasmax.nc
[########################################] | 100% Completed | 4hr 58m
Done.


# Compute tasmean from tasmax and tasmin
- Using cdo
- (tasmax+tasmin)/2

In [5]:
# 8 threads decompress/compute blocks in parallel; raise num_workers or MULT for more RAM/speed.
dask.config.set(scheduler="threads", num_workers=16)

OUTPUT_DIR = Path.cwd() / "data" / "CHELSA-TraCE21k-centennial" / "output"
tasmin_nc = OUTPUT_DIR / "tasmin.nc"
tasmax_nc = OUTPUT_DIR / "tasmax.nc"
tasmean_nc = OUTPUT_DIR / "tasmean.nc"
tasmean_nc.unlink(missing_ok=True)

# chunks={} inherits the file's on-disk chunking so reads stay aligned.
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
tmin = xr.open_dataset(tasmin_nc, chunks={}, decode_times=time_coder)
tmax = xr.open_dataset(tasmax_nc, chunks={}, decode_times=time_coder)

# Group whole on-disk chunks into bigger work units.
# A whole multiple of the native chunk never splits one, so reads stay aligned. Bigger MULT = fewer blocks, more RAM, faster.
MULT = 2
native_chunks = {dim: sizes[0] for dim, sizes in tmin["tasmin"].chunksizes.items()}
block = {dim: size * MULT for dim, size in native_chunks.items()}
block["time"] = 1
a = tmin["tasmin"].chunk(block)
b = tmax["tasmax"].chunk(block)

# Lazy recipe: nothing computed yet. Both inputs are in Kelvin, so it's a plain mean.
tasmean = ((a + b) * 0.5).astype("float32").rename("tasmean")
tasmean.attrs = {
    "long_name": "Yearly mean of daily mean near-surface air temperature",
    "units": "K",
    "source": "CHELSA-TraCE21k v1.0 (centennial)",
    "comment": "tasmean = (tasmin + tasmax) / 2, per centennial timestep.",
}

out = tasmean.to_dataset()

# Keep the CRS info (rioxarray stores it in the 'spatial_ref' coordinate).
if "spatial_ref" in tmin.coords:
    out = out.assign_coords(spatial_ref=tmin["spatial_ref"])
    out["tasmean"].attrs["grid_mapping"] = "spatial_ref"

# Writing runs the graph block by block, streaming to disk.
encoding = {"tasmean": {"zlib": True, "complevel": 4, "dtype": "float32"}}

with TqdmCallback(desc="Computing tasmean:"):
    out.to_netcdf(tasmean_nc, encoding=encoding, unlimited_dims=["time"])

print(f"Done -> {tasmean_nc} (exists: {tasmean_nc.exists()})")

Computing tasmean::   0%|          | 0/141883 [00:00<?, ?it/s]

Done -> /media/eskilsg/INTENSO/paleoclimate_model_comparison/data/CHELSA-TraCE21k-centennial/output/tasmean.nc (exists: True)


## PalMod2: Yearly Mean Computation & Merge

Computes annual means from monthly NetCDF files and merges them into a single output file using **CDO** (Climate Data Operators).

**Steps:**
1. Collect all `.nc` files from `INPUT_DIR`.
2. For each file: apply `cdo yearmean` (collapses 12 monthly values → 1 yearly mean) and save to a temporary directory.
3. Merge all yearly-mean files along the time axis with `cdo mergetime` into a single output file.

**Notes:**
- Temporary files are auto-deleted after the cell finishes.
- `tqdm` shows progress for the per-file step (the slow part).
- Time axis runs forward in model years (oldest → youngest), as the simulation integrates forward from the Last Glacial Maximum.

In [6]:
import subprocess
import shutil
import tarfile
from pathlib import Path
from tempfile import TemporaryDirectory
from tqdm import tqdm

#DATA_DIR = Path("/home/luser/Documents/Uni/TU/4 Semester/Geoinformatics Project/paleoclimate_model_comparison/data/PalMod2/data")
DATA_DIR = Path.cwd() / "data" / "PalMod2" / "data"
OUTPUT_DIR = DATA_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = [
    "PMMXMCRTDGr111Amtasgn30201_1-250",
    "PMMXMCRTDGr111Lmtslgn30201_1-250",
]


def process_dataset(name):
    extract_dir = DATA_DIR / name
    output_nc = OUTPUT_DIR / f"{name}.nc"
    print(f"Processing: {name}")

    with tarfile.open(DATA_DIR / f"{name}.tar") as tar:
        tar.extractall(extract_dir, filter="data")

    nc_files = sorted(extract_dir.glob("**/*.nc"))
    print(f"Found {len(nc_files)} files.")

    with TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        yearmean_files = []
        for f in tqdm(nc_files, desc="Computing yearly means"):
            out = tmp / f"{f.stem}_ymean.nc"
            # cwd + relative name so CDO never sees the space-containing path
            subprocess.run(["cdo", "yearmean", str(f.relative_to(extract_dir)), str(out)],
                           check=True, cwd=extract_dir)
            yearmean_files.append(str(out))

        print("Merging...")
        merged = tmp / f"{name}.nc"  # merge in the temp dir (no spaces), then move
        subprocess.run(["cdo", "mergetime", *yearmean_files, str(merged)], check=True)
        shutil.move(str(merged), str(output_nc))

    print(f"Done: {output_nc}")


for name in datasets:
    process_dataset(name)

Processing: PMMXMCRTDGr111Amtasgn30201_1-250
Found 250 files.


Computing yearly means:   1%|▌                                                               | 2/250 [00:00<00:47,  5.19it/s]

cdo    yearmean: Processed 5529600 values from 1 variable over 1299 timesteps [0.12s 14GB].
cdo    yearmean: Processed 5529600 values from 1 variable over 1299 timesteps [0.14s 14GB].


Computing yearly means:   2%|█                                                               | 4/250 [00:00<00:41,  5.89it/s]

cdo    yearmean: Processed 5529600 values from 1 variable over 1299 timesteps [0.14s 14GB].
cdo    yearmean: Processed 5529600 values from 1 variable over 1299 timesteps [0.14s 14GB].


...

Computing yearly means: 100%|█████████████████████████████████████████████████████████████▊| 249/250 [02:25<00:00,  1.72it/s]

cdo    yearmean: Processed 27648000 values from 1 variable over 1299 timesteps [0.56s 14GB].


Computing yearly means: 100%|██████████████████████████████████████████████████████████████| 250/250 [02:25<00:00,  1.72it/s]

cdo    yearmean: Processed 27648000 values from 1 variable over 1299 timesteps [0.56s 14GB].
Merging...


cdo    mergetime: Processed 576000000 values from 250 variables over 25000 timesteps [4.58s 14GB].
Done: /media/eskilsg/INTENSO/paleoclimate_model_comparison/data/PalMod2/output/PMMXMCRTDGr111Lmtslgn30201_1-250.nc
